In [ ]:
# Load data: order lines with order, itemtype, quantity, tote
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

from ranDataGen.order_data_loader import load_order_data

size = "Simplified-Dataset"  # Used for output paths
sample = 1  # Dataset sample index for output filenames

# ================================
# FOR SIMPLIFIED DATASET
# ================================

df = load_order_data(
    itemtypes_path="order_itemtypes.csv",
    quantities_path="order_quantities.csv",
    totes_path="orders_totes.csv",
    base_dir="ranDataGen/",
)

# ================================
# FOR SIMULATIONS
# ================================

# base_dir = f"ranDataGen copy/{size} sized samples/"

# itemtypes_path = f"order_itemtypes_{sample}.csv"
# quantities_path = f"order_quantities_{sample}.csv"
# totes_path = f"orders_totes_{sample}.csv"

# Load baseline data using modular loader
# df = load_order_data(
#     itemtypes_path=itemtypes_path,
#     quantities_path=quantities_path,
#     totes_path=totes_path,
#     base_dir=base_dir,
# )

df

,order,item_slot,itemtype,quantity,tote
0,1,0,3,3,1
1,1,1,1,2,1
2,2,0,2,3,2
3,2,1,3,1,3
4,2,2,0,1,2
5,3,0,3,3,3
6,3,1,2,3,2
7,3,2,0,1,1
8,4,0,1,1,0
9,4,1,2,1,0


# Time Based Model

In [ ]:

import gurobipy as gp
from gurobipy import GRB

m_cascade = gp.Model("MIP_greedy_cascade")  # Cascade model: upstream belts get first pick


# --------------------------------------------------
# Build item sequence (tote-contiguous order on conveyor)
# --------------------------------------------------


items = []  # List of (item_id, item_type, tote) for each physical unit
tote_items = {}  # tote_id -> list of (iid, itype, tote) in that tote

item_id = 0
for _, row in df.iterrows():
    itype = int(row["itemtype"])
    tote = int(row["tote"])
    qty = int(row["quantity"])

    for _ in range(qty):
        item_id += 1
        item = (item_id, itype, tote)
        items.append(item)
        tote_items.setdefault(tote, []).append(item)

items_seq = []  # Conveyor order: totes sorted by id, items contiguous within tote
for tote in sorted(tote_items.keys()):
    for (iid, itype, t) in tote_items[tote]:
        items_seq.append((iid, itype, t))

N = len(items_seq)  # Total number of items on conveyor
positions = range(1, N + 1)

pos_of_iid = {}  # item_id -> conveyor position 1..N
type_of_iid = {}  # item_id -> item type
iid_at_pos = {}  # position k -> item_id at that position

for p, (iid, itype, tote) in enumerate(items_seq, start=1):
    pos_of_iid[iid] = p
    type_of_iid[iid] = itype
    iid_at_pos[p] = iid

# --------------------------------------------------
# Sets and demand 
# --------------------------------------------------

belts = range(1, 5)  # Four belts in cascade order
orders = list(df["order"].unique())
item_types = sorted(df["itemtype"].unique())

Demand = {}  # (order, itemtype) -> quantity needed
for _, row in df.iterrows():
    o = int(row["order"])
    itype = int(row["itemtype"])
    qty = int(row["quantity"])
    if qty <= 0:
        continue
    Demand[(o, itype)] = Demand.get((o, itype), 0) + qty

max_dem = {(o, t): Demand.get((o, t), 0) for o in orders for t in item_types}  # Upper bounds for rem

# --------------------------------------------------
# Decision variables
# --------------------------------------------------

# y_c[b,o]=1 if order o is assigned to belt b
y_c = m_cascade.addVars(
    [(b, o) for b in belts for o in orders],
    vtype=GRB.BINARY,
    name="assign_c",
)

# Each order assigned to exactly one belt
m_cascade.addConstrs(
    (gp.quicksum(y_c[b, o] for b in belts) == 1 for o in orders),
    name="assign_one_belt_c",
)

# w[b,iid,o]=1 if belt b picks item iid for order o
w = m_cascade.addVars(
    [(b, iid, o) for b in belts for (iid, _, _) in items_seq for o in orders],
    vtype=GRB.BINARY,
    name="pick_c",
)

m_cascade.addConstrs(
    (
        gp.quicksum(w[b, iid, o] for b in belts for o in orders) <= 1
        for (iid, _, _) in items_seq
    ),
    name="use_item_once_c",  # Each physical item used at most once
)

m_cascade.addConstrs(
    (w[b, iid, o] <= y_c[b, o] for b in belts for (iid, _, _) in items_seq for o in orders),
    name="pick_implies_assign_c",  # Can only pick for an order if that order is on this belt
)

valid_pairs = set(Demand.keys())  # (order, itemtype) with positive demand 
for (iid, itype, _) in items_seq:
    for o in orders:
        if (o, itype) not in valid_pairs:
            for b in belts:
                w[b, iid, o].ub = 0  # Disallow pick when order doesn't need this type

for (o, itype), qty in Demand.items():  # Satisfy exact demand per (order, type)
    m_cascade.addConstr(
        gp.quicksum(
            w[b, iid, o]
            for b in belts
            for (iid, it, _) in items_seq
            if it == itype
        )
        == qty,
        name=f"demand_c_o{o}_t{itype}",
    )

# --------------------------------------------------
# Order sequencing on each belt (no interleaving: finish one order before starting next)
# --------------------------------------------------
# f_pos[b,o] = first conveyor position where belt b picks for order o
f_pos = m_cascade.addVars(
    [(b, o) for b in belts for o in orders],
    vtype=GRB.CONTINUOUS,
    lb=1.0,
    ub=N,
    name="first_pos",
)
# l_pos[b,o] = last conveyor position where belt b picks for order o
l_pos = m_cascade.addVars(
    [(b, o) for b in belts for o in orders],
    vtype=GRB.CONTINUOUS,
    lb=1.0,
    ub=N,
    name="last_pos",
)

BigN = float(N)  # Big-M for position constraints

# If belt b picks order o at position k, then f_pos[b,o] <= k and l_pos[b,o] >= k
for b in belts:
    for o in orders:
        for k in range(1, N + 1):
            iid_k = iid_at_pos[k]
            m_cascade.addConstr(
                f_pos[b, o] <= k + BigN * (1 - w[b, iid_k, o]),
                name=f"firstpos_ub_b{b}_o{o}_k{k}",
            )
            m_cascade.addConstr(
                l_pos[b, o] >= k - BigN * (1 - w[b, iid_k, o]),
                name=f"lastpos_lb_b{b}_o{o}_k{k}",
            )

# For each belt, all unordered pairs of orders (to enforce non-overlapping intervals)
order_pairs = [
    (b, o1, o2)
    for b in belts
    for i, o1 in enumerate(orders)
    for o2 in orders[i + 1 :]
]
# before[b,o1,o2]=1 if on belt b order o1 is completed before o2 starts
before = m_cascade.addVars(order_pairs, vtype=GRB.BINARY, name="order_before")

for b, o1, o2 in order_pairs:
    m_cascade.addConstr(
        l_pos[b, o1]
        <= f_pos[b, o2]
        + BigN
        * (1 - before[b, o1, o2] + (1 - y_c[b, o1]) + (1 - y_c[b, o2])),
        name=f"no_overlap1_b{b}_o{o1}_o{o2}",
    )
    m_cascade.addConstr(
        l_pos[b, o2]
        <= f_pos[b, o1]
        + BigN
        * (before[b, o1, o2] + (1 - y_c[b, o1]) + (1 - y_c[b, o2])),
        name=f"no_overlap2_b{b}_o{o1}_o{o2}",
    )

# --------------------------------------------------
# Remaining-demand flow along the conveyor (tracks how much each belt/order still needs by position)
# --------------------------------------------------
# rem[b,o,t,k] = remaining demand of type t for order o on belt b after processing positions 1..k
rem = {}
# has_rem[b,o,t,k] = 1 if rem[b,o,t,k] > 0 (used for greedy cascade)
has_rem = {}

for b in belts:
    for o in orders:
        for t in item_types:
            if max_dem[(o, t)] == 0:
                continue
            for k in range(0, N + 1):
                rem[b, o, t, k] = m_cascade.addVar(
                    vtype=GRB.INTEGER,
                    lb=0,
                    ub=max_dem[(o, t)],
                    name=f"rem_b{b}_o{o}_t{t}_k{k}",
                )
                if k < N + 1:
                    has_rem[b, o, t, k] = m_cascade.addVar(
                        vtype=GRB.BINARY,
                        name=f"hasrem_b{b}_o{o}_t{t}_k{k}",
                    )

m_cascade.update()

for b in belts:
    for o in orders:
        for t in item_types:
            if max_dem[(o, t)] == 0:
                continue
            d_ot = max_dem[(o, t)]

            m_cascade.addConstr(
                rem[b, o, t, 0] <= d_ot * y_c[b, o],
                name=f"rem0_ub_b{b}_o{o}_t{t}",
            )
            m_cascade.addConstr(
                rem[b, o, t, 0] >= d_ot * y_c[b, o] - d_ot * (1 - y_c[b, o]),
                name=f"rem0_lb_b{b}_o{o}_t{t}",
            )

for b in belts:
    for o in orders:
        for t in item_types:
            if max_dem[(o, t)] == 0:
                continue
            d_ot = max_dem[(o, t)]
            for k in range(1, N + 1):
                iid_k = iid_at_pos[k]
                itype_k = type_of_iid[iid_k]
                pick_here = w[b, iid_k, o] if itype_k == t else 0

                if itype_k == t:
                    m_cascade.addConstr(
                        rem[b, o, t, k]
                        == rem[b, o, t, k - 1] - w[b, iid_k, o],
                        name=f"rem_flow_b{b}_o{o}_t{t}_k{k}",
                    )
                else:
                    m_cascade.addConstr(
                        rem[b, o, t, k] == rem[b, o, t, k - 1],
                        name=f"rem_flow_b{b}_o{o}_t{t}_k{k}",
                    )

                # Link rem and has_rem: has_rem=1 iff rem > 0
                m_cascade.addConstr(
                    rem[b, o, t, k - 1]
                    <= d_ot * has_rem[b, o, t, k - 1],
                    name=f"hasrem_ub_b{b}_o{o}_t{t}_k{k-1}",
                )
                m_cascade.addConstr(
                    rem[b, o, t, k - 1]
                    >= has_rem[b, o, t, k - 1],
                    name=f"hasrem_lb_b{b}_o{o}_t{t}_k{k-1}",
                )

# --------------------------------------------------
# Greedy cascade:
# --------------------------------------------------
# U[b,k]=1 if any upstream belt (<b) still has remaining demand for the type at position k
U = {} 

for b in belts:
    if b == 1:
        continue
    for k in range(1, N + 1):
        iid_k = iid_at_pos[k]
        t_k = type_of_iid[iid_k]
        if max_dem.get((orders[0], t_k), 0) is None:
            continue
        U[b, k] = m_cascade.addVar(
            vtype=GRB.BINARY, name=f"upstream_need_b{b}_k{k}"
        )

m_cascade.update()

for b in belts:
    if b == 1:
        continue
    for k in range(1, N + 1):
        iid_k = iid_at_pos[k]
        t_k = type_of_iid[iid_k]

        if all(max_dem[(o, t_k)] == 0 for o in orders):
            continue

        for bp in belts:
            if bp >= b:
                continue
            for o in orders:
                if max_dem[(o, t_k)] == 0:
                    continue
                m_cascade.addConstr(
                    U[b, k] >= has_rem[bp, o, t_k, k - 1],
                    name=f"upstream_flag_b{b}_bp{bp}_o{o}_k{k}",
                )
        # If any upstream belt still needs type at k-1, belt b cannot pick this item
        m_cascade.addConstr(
            gp.quicksum(w[b, iid_k, o] for o in orders) <= 1 - U[b, k],
            name=f"no_downstream_pick_b{b}_k{k}",
        )

# --------------------------------------------------
# Time-based objective: minimize time when last pick completes
# --------------------------------------------------
TIME_c = m_cascade.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="completion_time_c")
# Pick time formula: position and belt offset; Mtime is big-M when pick=0
Mtime = 4.5 * N + 9.5 * (max(belts) - 1) + 3.5

for b in belts:
    for (iid, _, _) in items_seq:
        p = pos_of_iid[iid]
        for o in orders:
            m_cascade.addConstr(
                TIME_c
                >= 4.5 * p + 9.5 * (b - 1) + 3.5 - Mtime * (1 - w[b, iid, o]),
                name=f"time_ge_pick_c_b{b}_i{iid}_o{o}",
            )

m_cascade.setObjective(TIME_c, GRB.MINIMIZE)
m_cascade.Params.TimeLimit = 60  # Seconds
m_cascade.Params.MIPGap = 0.05  # Stop if within 5% of optimal
m_cascade.optimize()

# ==================================================
# Print Results (sequence, belt assignments, picks with time, completion time)
# ==================================================
from pathlib import Path
out_dir = Path("MIP") / "output" / str(size)
out_dir.mkdir(parents=True, exist_ok=True)
output_txt_path = out_dir / f"model_output_cascade_{size}_{sample}.txt"
_output_lines = []

def log(s: str = ""):
    print(s)
    _output_lines.append(str(s))

log("\nITEM SEQUENCE ON CONVEYOR")

item_type_names = {
    0: "circle",
    1: "pentagon",
    2: "trapezoid",
    3: "triangle",
    4: "star",
    5: "moon",
    6: "heart",
    7: "cross",
}

sequence = []
for p in positions:
    iid, itype, tote = items_seq[p - 1]
    sequence.append((p, iid, itype, tote))
sequence.sort()
# Log conveyor sequence by position
for p, iid, itype, tote in sequence:
    name = item_type_names.get(itype, f"type-{itype}")
    log(f"Position {p}: Item {iid} | Type {itype} ({name}) | Tote {tote}")

log("\nBELT → ORDER ASSIGNMENTS (cascade)")
for b in belts:
    for o in orders:
        if y_c[b, o].X > 0.5:
            log(f"Belt {b} processes Order {o}")

log("\nPICKS WITH TIME (cascade)")

picks_with_time = []
for b in belts:
    for (iid, itype, tote) in items_seq:
        for o in orders:
            if w[b, iid, o].X > 0.5:
                p = pos_of_iid[iid]
                t_pick = 4.5 * p + 9.5 * (b - 1) + 3.5
                picks_with_time.append((t_pick, b, iid, itype, tote, o))

picks_with_time.sort(key=lambda x: x[0])
for t_pick, b, iid, itype, tote, o in picks_with_time:
    log(
        f"Belt {b} picks Item {iid} (Type {itype}, Tote {tote}) for Order {o} → Time = {t_pick:.2f} sec"
    )

if m_cascade.SolCount > 0:
    log(f"\nSYSTEM COMPLETION TIME (cascade): {TIME_c.X}")

output_txt_path.write_text("\n".join(_output_lines) + "\n", encoding="utf-8")
log(f"\nCascade model output saved: {output_txt_path}")

Set parameter TimeLimit to value 60
Set parameter MIPGap to value 0.05
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[x86] - Darwin 24.6.0 24G517)

CPU model: Intel(R) Core(TM) i7-1060NG7 CPU @ 1.20GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  60
MIPGap  0.05

Optimize a model with 4026 rows, 2034 columns and 9037 nonzeros
Model fingerprint: 0x6b605ff3
Variable types: 33 continuous, 2001 integer (1201 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+02]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 2e+01]
  RHS range        [1e+00, 1e+02]
Presolve removed 3177 rows and 1661 columns
Presolve time: 0.10s
Presolved: 849 rows, 373 columns, 2750 nonzeros
Variable types: 0 continuous, 373 integer (312 binary)
Found heuristic solution: objective 117.5000000

Root relaxation: objective 7.190000e+01, 306 iterations, 0.01 seconds (0.01 work units)

    Nodes    |    Current Node    |     Ob

# CSV Input File

In [21]:
# Map item type id to shape name for CSV columns
shape_map = {
    0: "circle",
    1: "pentagon",
    2: "trapezoid",
    3: "triangle",
    4: "star",
    5: "moon",
    6: "heart",
    7: "cross"
}

belts = range(1, 5)
rows = []

# One row per belt: conv_num and count of each shape picked on that belt
for b in belts:
    row = {"conv_num": b}
    for col in shape_map.values():
        row[col] = 0
    for item_id, item_type, tote in items:

        if item_type not in shape_map:
            continue

        shape_name = shape_map[item_type]

        count = 0

        for o in orders:
            if w[b, item_id, o].X > 0.5:  # Count if this item picked for any order on belt b
                count += 1
        row[shape_name] += count
    rows.append(row)

df_picklist = pd.DataFrame(rows)
df_picklist = df_picklist[["conv_num"] + list(shape_map.values())]

from pathlib import Path
out_dir = Path("MIP") / "output" / str(size)
out_dir.mkdir(parents=True, exist_ok=True)
output_path = out_dir / f"MIP-belt_picklist_{size}_{sample}.csv"
df_picklist.to_csv(output_path, index=False)
print("CSV generated:", output_path)

CSV generated: MIP/output/Simplified-Dataset/MIP-belt_picklist_Simplified-Dataset_1.csv
